# Custom GAN — PyTorch

**Goal:** Build a GAN from scratch with PyTorch, trained exclusively on stroke cases.

Instead of a ready-made library (SDV), here we have full control over the architecture and the training loop — Generator vs Discriminator.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, average_precision_score, matthews_corrcoef, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid')

DATA_PATH = Path('../../data/healthcare-dataset-stroke-data.csv')
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load & Split

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other'].copy()
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['stroke']
)

print(f'Train: {train_df.shape} | Stroke: {train_df["stroke"].sum()}')
print(f'Test:  {test_df.shape}  | Stroke: {test_df["stroke"].sum()}')

## 2. Preprocessing for the GAN

The GAN requires **numerical** data in a **fixed range**. We use:
- `OrdinalEncoder` for categoricals (strings → integers)
- `MinMaxScaler` for all features → range [0, 1]

**Why MinMaxScaler instead of StandardScaler?**  
The Generator uses `sigmoid` as its output activation — which produces values in [0, 1]. MinMaxScaler aligns naturally with this range.

In [ ]:
NUMERICAL   = ['age', 'avg_glucose_level', 'bmi']
CATEGORICAL = ['hypertension', 'heart_disease', 'gender', 'ever_married',
               'work_type', 'Residence_type', 'smoking_status']
FEATURES    = NUMERICAL + CATEGORICAL

# Encode categoricals
enc = OrdinalEncoder()
train_enc = train_df.copy()
test_enc  = test_df.copy()
train_enc[CATEGORICAL] = enc.fit_transform(train_df[CATEGORICAL])
test_enc[CATEGORICAL]  = enc.transform(test_df[CATEGORICAL])

# Extract minority class (stroke=1) from train
minority_train = train_enc[train_enc['stroke'] == 1][FEATURES].values

# MinMaxScaler — fit on minority train only
scaler = MinMaxScaler()
minority_scaled = scaler.fit_transform(minority_train)

n_majority    = (train_df['stroke'] == 0).sum()
n_minority    = (train_df['stroke'] == 1).sum()
n_to_generate = n_majority - n_minority
INPUT_DIM     = minority_scaled.shape[1]

print(f'Minority samples: {n_minority}')
print(f'Samples to generate: {n_to_generate}')
print(f'Feature dimension: {INPUT_DIM}')

## 3. GAN Architecture

A GAN has two networks trained simultaneously:

**Generator:** Takes random noise (e.g. 100 numbers from a Gaussian) and transforms it into a synthetic sample that resembles a real stroke case.

**Discriminator:** Takes a sample (real or fake) and decides whether it is genuine or generated.

Training is a "game":
- The Discriminator learns to tell real from fake
- The Generator learns to fool the Discriminator

In [ ]:
NOISE_DIM = 100

class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Sigmoid()  # output in [0,1] — compatible with MinMaxScaler
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()  # output: probability of being real [0,1]
        )

    def forward(self, x):
        return self.net(x)


G = Generator(NOISE_DIM, INPUT_DIM).to(DEVICE)
D = Discriminator(INPUT_DIM).to(DEVICE)

print('Generator:')
print(G)
print('\nDiscriminator:')
print(D)

## 4. Training

**Loss function:** Binary Cross Entropy — the Discriminator wants to output 1 for real samples and 0 for fake. The Generator wants the Discriminator to output 1 for the fakes it produces.

**Each epoch:**
1. Train Discriminator: sees real + fake samples, learns to distinguish them
2. Train Generator: produces fake samples, tries to fool the Discriminator

In [ ]:
EPOCHS     = 500
BATCH_SIZE = 32
LR         = 0.0002

criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

# DataLoader for minority samples
X_tensor = torch.FloatTensor(minority_scaled)
loader   = DataLoader(TensorDataset(X_tensor), batch_size=BATCH_SIZE, shuffle=True)

g_losses, d_losses = [], []

for epoch in range(EPOCHS):
    g_loss_epoch, d_loss_epoch = 0, 0

    for (real_batch,) in loader:
        real_batch = real_batch.to(DEVICE)
        bs = real_batch.size(0)

        real_labels = torch.ones(bs, 1).to(DEVICE)
        fake_labels = torch.zeros(bs, 1).to(DEVICE)

        # --- Train Discriminator ---
        opt_D.zero_grad()
        loss_real = criterion(D(real_batch), real_labels)
        z = torch.randn(bs, NOISE_DIM).to(DEVICE)
        fake = G(z).detach()  # detach: no gradients through G here
        loss_fake = criterion(D(fake), fake_labels)
        loss_D = (loss_real + loss_fake) / 2
        loss_D.backward()
        opt_D.step()

        # --- Train Generator ---
        opt_G.zero_grad()
        z = torch.randn(bs, NOISE_DIM).to(DEVICE)
        fake = G(z)
        # G wants D to classify fakes as real
        loss_G = criterion(D(fake), real_labels)
        loss_G.backward()
        opt_G.step()

        g_loss_epoch += loss_G.item()
        d_loss_epoch += loss_D.item()

    g_losses.append(g_loss_epoch / len(loader))
    d_losses.append(d_loss_epoch / len(loader))

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss D: {d_losses[-1]:.4f} | Loss G: {g_losses[-1]:.4f}')

print('\nTraining complete.')

### 4.1 Loss Curves

In a well-trained GAN both losses converge around 0.5 — meaning the Discriminator can no longer distinguish fake from real (random 50/50 guess).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g_losses, label='Generator Loss',     color='#F44336', alpha=0.8)
ax.plot(d_losses, label='Discriminator Loss', color='#2196F3', alpha=0.8)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Equilibrium (0.5)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('GAN Training Loss', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../../data/gan_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Generate Synthetic Samples

The Generator outputs values in [0, 1] (due to sigmoid). We need to convert them back:
1. **Inverse MinMaxScaler** → original scales
2. **Rounding** for categorical features → valid integers

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(n_to_generate, NOISE_DIM).to(DEVICE)
    synthetic_scaled = G(z).cpu().numpy()

# Inverse transform → original scales
synthetic_raw = scaler.inverse_transform(synthetic_scaled)

# Numerical: clip to valid ranges
synthetic_raw[:, 0] = np.clip(synthetic_raw[:, 0], 0, 100)    # age
synthetic_raw[:, 1] = np.clip(synthetic_raw[:, 1], 50, 300)   # avg_glucose_level
synthetic_raw[:, 2] = np.clip(synthetic_raw[:, 2], 10, 100)   # bmi

# Categorical: round to nearest integer + clip to valid range
cat_ranges = {
    3: (0, 1),  # hypertension
    4: (0, 1),  # heart_disease
    5: (0, 1),  # gender (2 categories → 0,1)
    6: (0, 1),  # ever_married
    7: (0, 4),  # work_type (5 categories → 0-4)
    8: (0, 1),  # Residence_type
    9: (0, 3),  # smoking_status (4 categories → 0-3)
}
for col_idx, (min_val, max_val) in cat_ranges.items():
    synthetic_raw[:, col_idx] = np.clip(
        np.round(synthetic_raw[:, col_idx]), min_val, max_val
    )

synthetic_df = pd.DataFrame(synthetic_raw, columns=FEATURES)
synthetic_df['stroke'] = 1

print(f'Generated {len(synthetic_df)} synthetic stroke samples')
synthetic_df.head(3)

## 6. Quality Check

In [ ]:
minority_train_enc = train_enc[train_enc['stroke'] == 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, NUMERICAL):
    ax.hist(minority_train_enc[feat], bins=25, alpha=0.6, label='Real',       color='#2196F3', density=True)
    ax.hist(synthetic_df[feat],       bins=25, alpha=0.6, label='Custom GAN', color='#FF9800', density=True)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Custom GAN — Distribution Comparison (Numerical)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../data/custom_gan_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Statistics Comparison ===')
for feat in NUMERICAL:
    print(f'{feat}:')
    print(f'  Real:       mean={minority_train_enc[feat].mean():.2f}, std={minority_train_enc[feat].std():.2f}')
    print(f'  Custom GAN: mean={synthetic_df[feat].mean():.2f}, std={synthetic_df[feat].std():.2f}')

## 7. Augment & Classify

In [ ]:
# Combine real train data + synthetic
train_augmented = pd.concat([
    train_enc[FEATURES + ['stroke']],
    synthetic_df
], ignore_index=True)

X_train_aug = train_augmented[FEATURES].values
y_train_aug = train_augmented['stroke'].values
X_test_full = test_enc[FEATURES].values
y_test      = test_enc['stroke'].values

# StandardScaler on numericals for classifiers
sc = StandardScaler()
n  = len(NUMERICAL)
X_train_s = np.hstack([sc.fit_transform(X_train_aug[:, :n]), X_train_aug[:, n:]])
X_test_s  = np.hstack([sc.transform(X_test_full[:, :n]),     X_test_full[:, n:]])

print(f'Augmented train: {X_train_s.shape} | Stroke: {y_train_aug.sum()} ({y_train_aug.mean()*100:.1f}%)')

results = []
def evaluate(method, clf_name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    f1     = f1_score(y_te, y_pred)
    pr_auc = average_precision_score(y_te, y_prob)
    mcc    = matthews_corrcoef(y_te, y_pred)
    print(f'\n[{method}] {clf_name}')
    print(f'  F1: {f1:.4f} | PR-AUC: {pr_auc:.4f} | MCC: {mcc:.4f}')
    print(classification_report(y_te, y_pred, target_names=['No Stroke', 'Stroke']))
    results.append({'Method': method, 'Classifier': clf_name,
                    'F1': round(f1,4), 'PR-AUC': round(pr_auc,4), 'MCC': round(mcc,4)})

evaluate('Custom GAN', 'Logistic Regression',
         LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
         X_train_s, y_train_aug, X_test_s, y_test)

evaluate('Custom GAN', 'Random Forest',
         RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
         X_train_s, y_train_aug, X_test_s, y_test)

## 8. Results

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

metrics = ['F1', 'PR-AUC', 'MCC']
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, metric in zip(axes, metrics):
    ax.bar(results_df['Classifier'], results_df[metric],
           color=['#2196F3', '#F44336'], alpha=0.8)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('Custom GAN Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../data/custom_gan_results.png', dpi=150, bbox_inches='tight')
plt.show()